In [1]:
import numpy as np
from ainsworth_ieee import *
from read_image import image_array
from tqdm import tqdm

In [2]:
import torch
import torch.nn.functional as F

def set_padding(input_array, x_window_size, y_window_size):
    pad_x = (x_window_size - 1) // 2
    pad_y = (y_window_size - 1) // 2
    padded_tensor = F.pad(torch.tensor(input_array), (pad_y, pad_y, pad_x, pad_x), mode='constant', value=0)
    return padded_tensor.numpy()

In [3]:
C11 = np.load(r'C:\Users\Vision IAS\Desktop\work\SAR\Covariance\Covariance Matrices\C11.npy') 
C12 = np.load(r'C:\Users\Vision IAS\Desktop\work\SAR\Covariance\Covariance Matrices\C12.npy') 
C13 = np.load(r'C:\Users\Vision IAS\Desktop\work\SAR\Covariance\Covariance Matrices\C13.npy') 
C14 = np.load(r'C:\Users\Vision IAS\Desktop\work\SAR\Covariance\Covariance Matrices\C14.npy') 
C21 = np.load(r'C:\Users\Vision IAS\Desktop\work\SAR\Covariance\Covariance Matrices\C21.npy')
C22 = np.load(r'C:\Users\Vision IAS\Desktop\work\SAR\Covariance\Covariance Matrices\C22.npy')
C23 = np.load(r'C:\Users\Vision IAS\Desktop\work\SAR\Covariance\Covariance Matrices\C23.npy')
C24 = np.load(r'C:\Users\Vision IAS\Desktop\work\SAR\Covariance\Covariance Matrices\C24.npy')
C31 = np.load(r'C:\Users\Vision IAS\Desktop\work\SAR\Covariance\Covariance Matrices\C31.npy')
C32 = np.load(r'C:\Users\Vision IAS\Desktop\work\SAR\Covariance\Covariance Matrices\C32.npy')
C33 = np.load(r'C:\Users\Vision IAS\Desktop\work\SAR\Covariance\Covariance Matrices\C33.npy')
C34 = np.load(r'C:\Users\Vision IAS\Desktop\work\SAR\Covariance\Covariance Matrices\C34.npy')
C41 = np.load(r'C:\Users\Vision IAS\Desktop\work\SAR\Covariance\Covariance Matrices\C41.npy')
C42 = np.load(r'C:\Users\Vision IAS\Desktop\work\SAR\Covariance\Covariance Matrices\C42.npy')
C43 = np.load(r'C:\Users\Vision IAS\Desktop\work\SAR\Covariance\Covariance Matrices\C43.npy')
C44 = np.load(r'C:\Users\Vision IAS\Desktop\work\SAR\Covariance\Covariance Matrices\C44.npy')

In [4]:
import numpy as np
from tqdm import tqdm
from numpy.linalg import LinAlgError
np.set_printoptions(linewidth=1000)

def compute_sigma_1_matrix(u, v, w, z, alpha):
    sqrt_alpha = np.sqrt(alpha)
    arr = np.array([[1, -w, -v, v*w],
                    [-u / sqrt_alpha, 1 / sqrt_alpha, (u*v) / sqrt_alpha, -v / sqrt_alpha],
                    [-z / sqrt_alpha, (w*z) / sqrt_alpha, sqrt_alpha, -w / sqrt_alpha],
                    [(u*z), -z, -u, 1]])

    multiplier = 1 / (((u*w) - 1) * ((v*z)-1))  
    return arr * multiplier

def compute_alpha(covariance):
    x = covariance[1,2] / np.abs(covariance[1,2])
    y = np.sqrt(np.abs(covariance[1,1] / covariance[2,2]))
    return x*y

def compute_A(covariance):
     return (covariance[1,0] + covariance[2,0]) / 2

def compute_B(covariance):
     return (covariance[1,3] + covariance[2,3]) / 2

def compute_X_matrix(covariance, A, B):
    x_matrix = np.array([
          [covariance[2,0] - A],
          [covariance[1,0] - A],
          [covariance[2,3] - B], 
          [covariance[1,3] - B]])
    
    return x_matrix

def compute_zeta_matrix(covariance):
    zeta_matrix = np.array([
          [0, 0, covariance[3,0], covariance[0,0]],
          [covariance[0,0], covariance[3,0], 0, 0],
          [0, 0, covariance[3,3], covariance[0,3]],
          [covariance[0,3], covariance[3,3], 0, 0]])
    
    return zeta_matrix

def compute_tau_matrix(covariance):
    tau_matrix = np.array([
          [0, covariance[2,2], covariance[2,1], 0],
          [0, covariance[1,2], covariance[1,1], 0],
          [covariance[2,2], 0, 0, covariance[2,1]],
          [covariance[1,2], 0, 0, covariance[1,1]]])
    
    return tau_matrix



def parameter_updates(x, zeta, tau):
    X_Matrix_Need = np.concatenate([np.real(x), np.imag(x)])

    Zeta_Tau_Matrix_Need = np.concatenate([np.concatenate([np.real(zeta + tau), -1 * np.imag(zeta - tau)], axis=1),
                                           np.concatenate([np.imag(zeta + tau), np.real(zeta - tau)], axis=1)])

    delta_Solve = np.linalg.inv(Zeta_Tau_Matrix_Need) @ X_Matrix_Need

    delta_u = delta_Solve[0] + 1j * delta_Solve[4]
    delta_v = delta_Solve[1] + 1j * delta_Solve[5]
    delta_w = delta_Solve[2] + 1j * delta_Solve[6]
    delta_z = delta_Solve[3] + 1j * delta_Solve[7]

    return delta_u, delta_v, delta_w, delta_z


def ainsworth_cal(covar):
    u = 0
    v = 0
    w = 0
    z = 0
    gamma = 100
    tolerance = 1e-8
    i_iter = 0
    alpha = compute_alpha(covar)

    while i_iter < 12:
        sigma_1_matrix_i = compute_sigma_1_matrix(u, v, w, z, alpha)

        covariance_i = sigma_1_matrix_i @ covar @ np.conj(sigma_1_matrix_i).T
        A = compute_A(covariance_i)
        B = compute_B(covariance_i)

        X = compute_X_matrix(covariance_i, A, B)
        zeta = compute_zeta_matrix(covariance_i)
        tau = compute_tau_matrix(covariance_i)
        delta_u, delta_v, delta_w, delta_z = parameter_updates(X, zeta, tau)
        alpha_i = compute_alpha(covariance_i)

        u = (u + delta_u / np.sqrt(alpha))[0]
        v = (v + delta_v / np.sqrt(alpha))[0]
        w = (w + delta_w * np.sqrt(alpha))[0]
        z = (z + delta_z * np.sqrt(alpha))[0]
        alpha = alpha * alpha_i

        gamma = (max(abs(u), abs(v), abs(w), abs(z)))
        i_iter = i_iter + 1
    
    return sigma_1_matrix_i

In [5]:
window_size_x,window_size_y = 201, 201

HH = set_padding(image_array(r'C:\Users\Vision IAS\Desktop\work\SAR\Output\HH_calibrated.tiff'), window_size_x, window_size_y)
HV = set_padding(image_array(r'C:\Users\Vision IAS\Desktop\work\SAR\Output\HV_calibrated.tiff'), window_size_x, window_size_y)
VH = set_padding(image_array(r'C:\Users\Vision IAS\Desktop\work\SAR\Output\VH_calibrated.tiff'), window_size_x, window_size_y)
VV = set_padding(image_array(r'C:\Users\Vision IAS\Desktop\work\SAR\Output\VV_calibrated.tiff'), window_size_x, window_size_y)

In [6]:
HH.shape

(3200, 3200)

In [7]:
# # 2388,2129
# row_center = 2499
# col_center = 2229
# window_size = 201

# # Calculate start and end indices
# radius = window_size // 2  # Half the window size
# row_start = row_center - radius
# row_end = row_center + radius + 1  # Add 1 because Python slicing is exclusive
# col_start = col_center - radius
# col_end = col_center + radius + 1

# # Python slicing for the 201x201 window
# rows = slice(row_start, row_end)
# cols = slice(col_start, col_end)

# # Example usage in an array
# # HHu[rows, cols]
# print(f"Rows: {rows}, Columns: {cols}")


In [8]:
# from tqdm import tqdm
# import numpy as np

# HH_cal = np.zeros((3000, 3000), dtype=np.complex128)
# HV_cal = np.zeros((3000, 3000), dtype=np.complex128)
# VH_cal = np.zeros((3000, 3000), dtype=np.complex128)
# VV_cal = np.zeros((3000, 3000), dtype=np.complex128)
# def sliding_window(HH, HV, VH, VV, window_size_x, window_size_y, stride_x, stride_y):
#     result = []
#     rows, cols = HH.shape

#     # Calculate total iterations for progress bar
#     total_iterations = ((rows - window_size_y) // stride_y + 1) * ((cols - window_size_x) // stride_x + 1)
#     pbar = tqdm(total=total_iterations, desc="Sliding Window Progress")

#     for i in range(0, rows - window_size_y + 1, stride_y):
#         for j in range(0, cols - window_size_x + 1, stride_x):
#             hh = HH[i:i + window_size_y, j:j + window_size_x]
#             hv = HV[i:i + window_size_y, j:j + window_size_x]
#             vh = VH[i:i + window_size_y, j:j + window_size_x]
#             vv = VV[i:i + window_size_y, j:j + window_size_x]

#             c11 = C11[i:i + window_size_y, j:j + window_size_x].mean()
#             c12 = C12[i:i + window_size_y, j:j + window_size_x].mean()
#             c13 = C13[i:i + window_size_y, j:j + window_size_x].mean()
#             c14 = C14[i:i + window_size_y, j:j + window_size_x].mean()
#             c21 = C21[i:i + window_size_y, j:j + window_size_x].mean()
#             c22 = C22[i:i + window_size_y, j:j + window_size_x].mean()
#             c23 = C23[i:i + window_size_y, j:j + window_size_x].mean()
#             c24 = C24[i:i + window_size_y, j:j + window_size_x].mean()
#             c31 = C31[i:i + window_size_y, j:j + window_size_x].mean()
#             c32 = C32[i:i + window_size_y, j:j + window_size_x].mean()
#             c33 = C33[i:i + window_size_y, j:j + window_size_x].mean()
#             c34 = C34[i:i + window_size_y, j:j + window_size_x].mean()
#             c41 = C41[i:i + window_size_y, j:j + window_size_x].mean()
#             c42 = C42[i:i + window_size_y, j:j + window_size_x].mean()
#             c43 = C43[i:i + window_size_y, j:j + window_size_x].mean()
#             c44 = C44[i:i + window_size_y, j:j + window_size_x].mean()

#             covariance = np.array([[c11, c12, c13, c14, c21, c22, c23, c24, c31, c32, c33, c34, c41, c42, c43, c44]]).reshape((4, 4))

#             x = ainsworth_cal(covariance)
#             out = x @ np.array([[hh[100, 100], hv[100, 100], vh[100, 100], vv[100, 100]]]).T
#             result.append(out)
#             pbar.update(1)

#     pbar.close()
#     return result


In [ ]:
import numpy as np
from tqdm.auto import tqdm
import multiprocessing as mp
from multiprocessing import Queue, Manager
import os
import traceback
import sys

def process_window(input_queue, output_queue, error_queue, 
                   HH, HV, VH, VV, 
                   C11, C12, C13, C14,
                   C21, C22, C23, C24,
                   C31, C32, C33, C34,
                   C41, C42, C43, C44,
                   window_size_x, window_size_y):
    """
    Worker process for sliding window computation
    
    Reads window parameters from input queue and puts results in output queue
    """
    try:
        while True:
            # Get next task from input queue
            task = input_queue.get()
            
            # Sentinel value to stop the process
            if task is None:
                break
            
            # Unpack window coordinates
            i, j = task
            
            # Extract window regions
            try:
                hh = HH[i:i + window_size_y, j:j + window_size_x]
                hv = HV[i:i + window_size_y, j:j + window_size_x]
                vh = VH[i:i + window_size_y, j:j + window_size_x]
                vv = VV[i:i + window_size_y, j:j + window_size_x]
                
                # Calculate mean covariance matrix components
                c11 = C11[i:i + window_size_y, j:j + window_size_x].mean()
                c12 = C12[i:i + window_size_y, j:j + window_size_x].mean()
                c13 = C13[i:i + window_size_y, j:j + window_size_x].mean()
                c14 = C14[i:i + window_size_y, j:j + window_size_x].mean()
                c21 = C21[i:i + window_size_y, j:j + window_size_x].mean()
                c22 = C22[i:i + window_size_y, j:j + window_size_x].mean()
                c23 = C23[i:i + window_size_y, j:j + window_size_x].mean()
                c24 = C24[i:i + window_size_y, j:j + window_size_x].mean()
                c31 = C31[i:i + window_size_y, j:j + window_size_x].mean()
                c32 = C32[i:i + window_size_y, j:j + window_size_x].mean()
                c33 = C33[i:i + window_size_y, j:j + window_size_x].mean()
                c34 = C34[i:i + window_size_y, j:j + window_size_x].mean()
                c41 = C41[i:i + window_size_y, j:j + window_size_x].mean()
                c42 = C42[i:i + window_size_y, j:j + window_size_x].mean()
                c43 = C43[i:i + window_size_y, j:j + window_size_x].mean()
                c44 = C44[i:i + window_size_y, j:j + window_size_x].mean()
                
                # Create covariance matrix
                covariance = np.array([
                    [c11, c12, c13, c14], 
                    [c21, c22, c23, c24],
                    [c31, c32, c33, c34],
                    [c41, c42, c43, c44]
                ])
                
                # Perform calculation (assuming ainsworth_cal is defined)
                x = ainsworth_cal(covariance)
                
                # Process specific pixel (modify as needed)
                out = x @ np.array([[hh[100, 100], hv[100, 100], vh[100, 100], vv[100, 100]]]).T
                
                # Put result in output queue
                output_queue.put((i, j, out))
            
            except Exception as e:
                # Log any errors for this specific window
                error_queue.put((i, j, str(e), traceback.format_exc()))
    
    except Exception as e:
        # Catch and log any unexpected errors
        error_queue.put((None, None, str(e), traceback.format_exc()))

def sliding_window_multiprocessing(HH, HV, VH, VV, 
                                   C11, C12, C13, C14,
                                   C21, C22, C23, C24,
                                   C31, C32, C33, C34,
                                   C41, C42, C43, C44,
                                   window_size_x, window_size_y, 
                                   stride_x, stride_y,
                                   num_cores=None):
    """
    Robust multiprocessing sliding window function
    
    Args:
        Input matrices and covariance matrices
        window_size_x, window_size_y: Window dimensions
        stride_x, stride_y: Sliding window strides
        num_cores: Number of cores to use
    
    Returns:
        List of processed window results
    """
    # Determine number of cores
    if num_cores is None:
        num_cores = 2 # Cap at 50 or available cores
    
    # Get matrix dimensions
    rows, cols = HH.shape
    
    # Create multiprocessing queues
    manager = Manager()
    input_queue = manager.Queue()
    output_queue = manager.Queue()
    error_queue = manager.Queue()
    
    # Generate window coordinates
    window_coords = [
        (i, j) for i in range(0, rows - window_size_y + 1, stride_y)
        for j in range(0, cols - window_size_x + 1, stride_x)
    ]
    total_windows = len(window_coords)
    
    # Create and start worker processes
    processes = []
    for _ in range(num_cores):
        p = mp.Process(target=process_window, args=(
            input_queue, output_queue, error_queue,
            HH, HV, VH, VV, 
            C11, C12, C13, C14,
            C21, C22, C23, C24,
            C31, C32, C33, C34,
            C41, C42, C43, C44,
            window_size_x, window_size_y
        ))
        p.start()
        processes.append(p)
    
    # Put window coordinates into input queue
    for coord in window_coords:
        input_queue.put(coord)
    
    # Add sentinel values to stop processes
    for _ in range(num_cores):
        input_queue.put(None)
    
    # Collect results with progress tracking
    results = {}
    errors = []
    completed = 0
    
    # Use tqdm for progress tracking
    with tqdm(total=total_windows, desc="Processing Windows") as pbar:
        while completed < total_windows:
            # Try to get a result or error
            try:
                # Use a timeout to prevent hanging
                try:
                    # Check output queue first
                    if not output_queue.empty():
                        i, j, result = output_queue.get(timeout=1)
                        results[(i, j)] = result
                        completed += 1
                        pbar.update(1)
                    
                    # Check error queue
                    if not error_queue.empty():
                        error = error_queue.get(timeout=0.1)
                        errors.append(error)
                
                except Exception:
                    # If queues are empty, just pass
                    pass
            
            except KeyboardInterrupt:
                # Allow graceful interruption
                break
    
    # Terminate all processes
    for p in processes:
        p.terminate()
        p.join()
    
    # Log any errors
    if errors:
        print("Errors occurred during processing:", file=sys.stderr)
        for error in errors:
            print(f"Window {error[0]},{error[1]}: {error[2]}", file=sys.stderr)
            print(error[3], file=sys.stderr)
    
    # Convert results to list in original order
    ordered_results = [results.get((i, j)) for (i, j) in window_coords if (i, j) in results]
    
    return ordered_results

# Example usage remains the same as in previous version
def main():
    
    # Process windows
    results = sliding_window_multiprocessing(
        HH, HV, VH, VV, 
        C11, C12, C13, C14, 
        C21, C22, C23, C24,
        C31, C32, C33, C34, 
        C41, C42, C43, C44,
        window_size_x=201, 
        window_size_y=201, 
        stride_x=1, 
        stride_y=1
    )

if __name__ == '__main__':
    main()